In [ ]:
# Hazard Inventoy Taiwan - the search terms need to be uploaded alongside this for the code to work
# They are also contained within the supplimentary material


!pip install -q gnews beautifulsoup4 requests pandas openpyxl geopandas shapely matplotlib_scalebar contextily fuzzywuzzy python-Levenshtein tqdm

#import modules:
import pandas as pd, re
from gnews import GNews
from fuzzywuzzy import process
import geopandas as gpd
from shapely.geometry import Point
import contextily as ctx
import matplotlib.pyplot as plt
from tqdm import tqdm


# Upload the search dicionary:
xls = pd.ExcelFile("/content/All_Terms.xlsx") #open excel file containing search terms
print("Sheets found:", xls.sheet_names) #checking sheets are there

#loading names and terms:
hazard_terms = pd.read_excel(xls, 'Hazard')
places_df    = pd.read_excel(xls, 'Places')
roads_df     = pd.read_excel(xls, 'Roads')
rail_df      = pd.read_excel(xls, 'Railways')


#text preprocessing and combination of search terms:
def clean_query(term):
    if pd.isna(term): return "" #leave missing entries blank
    term = str(term) #ensure terms are treated as text.
    #remove control characters that can break RSS query URLs.
    term = re.sub(r"[\x00-\x1f\x7f]", " ", term)
    term = re.sub(r"[,|]", " ", term)  #swap separators for normal spaces
    term = re.sub(r"\s+", " ", term).strip()
    return term

hazard_terms['zh_term'] = hazard_terms['Search Term (Traditional Chinese)'].apply(clean_query) #clean chinese search terms
hazard_terms['en_term'] = hazard_terms['English Translation'].apply(clean_query) #clean english terms


#remove duplicates and empty values:
zh_terms = [t for t in hazard_terms['zh_term'].unique() if t] #keep only unique non-empty zh terms
en_terms = [t for t in hazard_terms['en_term'].unique() if t] #same for en terms
print(f"\nTotal Mandarin terms: {len(zh_terms)}, English terms: {len(en_terms)}")


#merge Places, Roads, and Railways into one location dataset:
places_df.rename(columns={'Chinese Name': 'Name', 'lat': 'Latitude', 'lon': 'Longitude'}, inplace=True) #place column consistent naming
roads_df.rename(columns={'road_name': 'Name', 'lat': 'Latitude', 'lon': 'Longitude'}, inplace=True)
rail_df.rename(columns={'railway_name': 'Name', 'lat': 'Latitude', 'lon': 'Longitude'}, inplace=True)

places_df['Feature_Type'] = 'Place' #labels these records as places
roads_df['Feature_Type'] = 'Road'
rail_df['Feature_Type']  = 'Railway'

location_terms = pd.concat([places_df, roads_df, rail_df], ignore_index=True) #joins into one location dictionary
print(f"Combined location entries: {len(location_terms)}") #chow many location records


#text Preprocessing of news article titles:
def clean_text(t):
    if pd.isna(t): return '' # missing title issue mitigation
    t = re.sub(r'[^\w\s]', '', str(t)) #remove punctuation
    return t.lower()


# match up locations from the article titles to the location dictionary
def find_location(text):
    match = process.extractOne(text, location_terms['Name']) #find closest location name using fuzzy matching
    if match and match[1] > 85: #similarity score (ranging from 0 to 100, where 100 is a perfect match and 0 is no similarity at all). set to keep only reasonably strong batches ( >85)
        loc = location_terms.loc[location_terms['Name'] == match[0]].iloc[0] #obtain matching location record
        return pd.Series([loc['Name'], loc['Latitude'], loc['Longitude'], loc['Feature_Type']]) # return matched values and type
    return pd.Series([None, None, None, None]) #leavr blank if no match.


#query GNews for Mandarin and English hazard terms:
MAX_RESULTS_PER_TERM = 1 #reduced to 1 for testing due to collab limits- increase later for full search

#set language and country:
gnews = GNews(language='zh', country='TW', max_results=MAX_RESULTS_PER_TERM) #search taiwan google news
all_results = []

print("\n Searching Mandarin hazard terms...")
for term in tqdm(zh_terms):
    try:
        results = gnews.get_news(term)
        for r in results:
            r['keyword'] = term
            r['lang'] = 'zh'
            all_results.append(r)
    except Exception as e:
        print(f" Chinese term '{term}' failed: {e}")

# Optional inclusion of english search:
#gnews_en = GNews(language='en', country='TW', max_results=MAX_RESULTS_PER_TERM)
#print("\n Searching for English hazard terms...")
#for term in tqdm(en_terms):
#    try:
#        results = gnews_en.get_news(term)
#        for r in results:
#            r['keyword'] = term
#            r['lang'] = 'en'
#            all_results.append(r)
#    except Exception as e:
#        print(f"English term '{term}' failed: {e}")


# combine and clean results:
news_df = pd.DataFrame(all_results) #data frame for collected news results
if len(news_df) == 0:
    raise ValueError("No articles retrieved. Try later or add more keywords.") #stop if nothing is returned.

news_df.rename(columns={'published date': 'date'}, inplace=True)
news_df['clean_text'] = news_df['title'].apply(clean_text) #clean article title for fuzzy matching.


# Fuzzy-match locations:
print("\n🗺 Matching locations (places, roads, railways)...")
news_df[['matched_name', 'latitude', 'longitude', 'feature_type']] = news_df['clean_text'].apply(find_location)


#build structured inventory including add in the lat + Longs:
news_df['hazard_type'] = news_df['keyword']
inventory = news_df[['date', 'title', 'description', 'hazard_type', 'lang',
                     'matched_name', 'feature_type', 'latitude', 'longitude', 'url']]
inventory.dropna(subset=['latitude'], inplace=True) #remove articles where location could not be matched.
inventory.to_csv('/content/taiwan_hazard_inventory.csv', index=False)

print(f"\n Saved {len(inventory)} geocoded hazard records to taiwan_hazard_inventory.csv")

#output map:
gdf = gpd.GeoDataFrame(
    inventory,
    geometry=gpd.points_from_xy(inventory.longitude, inventory.latitude), #convert coords into map points:
    crs="EPSG:4326" #uses the standard WGS 84 geographic coord system.
)

#plot records:
ax = gdf.plot(figsize=(8,8), alpha=0.7, markersize=40, column='feature_type', legend=True)
ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik)
plt.title('Text-Mined Multi-Feature Hazard Inventory of Taiwan (v4)')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.show()

Sheets found: ['Hazard', 'Feature', 'Impact', 'Buildings', 'Waterways', 'District_Township', 'Roads', 'Railways', 'Places']

Total Mandarin terms: 620, English terms: 622
Combined location entries: 162858

 Searching Mandarin hazard terms...


100%|██████████| 620/620 [04:44<00:00,  2.18it/s]



🗺 Matching locations (places, roads, railways)...

 Saved 27 geocoded hazard records to taiwan_hazard_inventory.csv


/tmp/ipykernel_2166/3003951575.py:128: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inventory.dropna(subset=['latitude'], inplace=True) #remove articles where location could not be matched.
